In [1]:
from pathlib import Path
from collections import Counter

import sys
import numpy as np
import pandas as pd
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)


print(
    "Python:",
    sys.executable
)

print(
    "Working directory:",
    Path.cwd()
)


# 1 Define project paths

BASE_DIR = Path.cwd()


# Primary final policy from Phase 17
POLICY_PATH = (
    BASE_DIR
    / "phase17_final_haptic_policy_rerun.csv"
)


# Fine-tuned ReadaBERT from Phase 5
READABERT_PATH = (
    BASE_DIR
    / "readabert_clear_finetuned_best_rerun"
)


# CLEAR training split used for Phase 5/6
CLEAR_TRAIN_PATH = Path(
    "/Users/nataliakolbina/Documents/workspace/haptic/"
    "CLEAR_original_grouped_split_2026-09-25/"
    "CLEAR_original_train.csv"
)


# Phase 7 gaze-model repository
GAZE_REPO = (
    BASE_DIR
    / "gaze-guided-text-generation"
)


GAZE_CHECKPOINT = (
    GAZE_REPO
    / "models"
    / "trf_gaze_model.pt"
)


# Phase 7 gaze predictions
GAZE_REFERENCE_PATH = (
    BASE_DIR
    / "clear_test_readabert_gaze_rerun.csv"
)


# Phase 9 state data
STATE_SOURCE_PATH = (
    BASE_DIR
    / "clear_text_states_phase9_rerun.csv"
)


# Phase 9 saved thresholds
THRESHOLD_PATH = (
    BASE_DIR
    / "phase9_discretisation_thresholds_rerun.csv"
)


# Phase 19 outputs
DEMO_OUTPUT_PATH = (
    BASE_DIR
    / "phase19_personalisation_demo_rerun.csv"
)


PROTOTYPE_OUTPUT_PATH = (
    BASE_DIR
    / "phase19_end_to_end_example_rerun.csv"
)


# 2 Check all required assets

required_assets = {
    "Final Phase 17 policy":
        POLICY_PATH,

    "Fine-tuned ReadaBERT":
        READABERT_PATH,

    "CLEAR training data":
        CLEAR_TRAIN_PATH,

    "Gaze repository":
        GAZE_REPO,

    "Gaze checkpoint":
        GAZE_CHECKPOINT,

    "Gaze reference data":
        GAZE_REFERENCE_PATH,

    "Phase 9 state data":
        STATE_SOURCE_PATH,

    "Phase 9 thresholds":
        THRESHOLD_PATH
}


print(
    "\nPHASE 19 ASSET CHECK"
)

print(
    "-" * 65
)


missing_assets = []


for name, path in required_assets.items():

    exists = (
        path.exists()
    )

    print(
        f"{name:28s}: "
        f"{'FOUND' if exists else 'NOT FOUND'}"
    )

    if exists:

        print(
            "   ",
            path
        )

    else:

        missing_assets.append(
            name
        )


if missing_assets:

    raise FileNotFoundError(
        "Missing Phase 19 assets: "
        + ", ".join(
            missing_assets
        )
    )


print(
    "\nAll required Phase 19 assets found."
)

# 3 Load the frozen primary Phase 17 policy

policy_df = pd.read_csv(
    POLICY_PATH
)


required_policy_columns = [
    "D_level",
    "C_level",
    "P_level",
    "Haptic_Action"
]


missing_policy_columns = [
    column
    for column in required_policy_columns
    if column not in policy_df.columns
]


if missing_policy_columns:

    raise ValueError(
        "Missing policy columns: "
        + ", ".join(
            missing_policy_columns
        )
    )


FINAL_POLICY = {}


for _, row in policy_df.iterrows():

    state = (
        int(
            row[
                "D_level"
            ]
        ),

        int(
            row[
                "C_level"
            ]
        ),

        int(
            row[
                "P_level"
            ]
        )
    )


    FINAL_POLICY[
        state
    ] = int(
        row[
            "Haptic_Action"
        ]
    )


HAPTIC_ACTIONS = {
    0: "Low",
    1: "Medium",
    2: "High"
}


LEVEL_NAMES = {
    0: "Low",
    1: "Medium",
    2: "High"
}


print(
    "\nFinal policy states:",
    len(
        FINAL_POLICY
    )
)


print(
    "Final policy action distribution:",
    Counter(
        FINAL_POLICY.values()
    )
)


if len(
    FINAL_POLICY
) != 27:

    raise ValueError(
        "Expected exactly 27 states "
        "in the frozen Phase 17 policy."
    )

# 4 Load the fine-tuned ReadaBERT model

readability_tokenizer = (
    AutoTokenizer.from_pretrained(
        str(
            READABERT_PATH
        ),
        local_files_only=True
    )
)


readability_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        str(
            READABERT_PATH
        ),
        local_files_only=True
    )
)


if torch.cuda.is_available():

    readability_device = (
        torch.device(
            "cuda"
        )
    )


elif (
    hasattr(
        torch.backends,
        "mps"
    )
    and
    torch.backends.mps.is_available()
):

    readability_device = (
        torch.device(
            "mps"
        )
    )


else:

    readability_device = (
        torch.device(
            "cpu"
        )
    )


readability_model.to(
    readability_device
)

readability_model.eval()


print(
    "\nReadaBERT loaded:"
)

print(
    READABERT_PATH
)

print(
    "ReadaBERT device:",
    readability_device
)

# 5 Freeze the Phase 6 easiness-to-difficulty scaling

clear_train = pd.read_csv(
    CLEAR_TRAIN_PATH
)


if "BT.Easiness" not in clear_train.columns:

    raise ValueError(
        "BT.Easiness is missing "
        "from the CLEAR training data."
    )


BT_MIN = float(
    clear_train[
        "BT.Easiness"
    ].min()
)


BT_MAX = float(
    clear_train[
        "BT.Easiness"
    ].max()
)


print(
    "\nPhase 6 BT.Easiness range:"
)

print(
    "Minimum:",
    BT_MIN
)

print(
    "Maximum:",
    BT_MAX
)

# 6 Freeze the Phase 8 gaze-score scaling

gaze_reference = pd.read_csv(
    GAZE_REFERENCE_PATH
)


if "Gaze_Score_Raw" not in gaze_reference.columns:

    raise ValueError(
        "Gaze_Score_Raw missing "
        "from Phase 7 output."
    )


GAZE_MIN = float(
    gaze_reference[
        "Gaze_Score_Raw"
    ].min()
)


GAZE_MAX = float(
    gaze_reference[
        "Gaze_Score_Raw"
    ].max()
)


print(
    "\nPhase 8 gaze-score range:"
)

print(
    "Minimum:",
    GAZE_MIN
)

print(
    "Maximum:",
    GAZE_MAX
)

# 7 Load the exact Phase 9 discretisation thresholds

threshold_df = pd.read_csv(
    THRESHOLD_PATH
)


print(
    "\nPhase 9 threshold file:"
)

print(
    threshold_df
)


if {
    "Signal",
    "Low_Medium_Threshold",
    "Medium_High_Threshold"
}.issubset(
    threshold_df.columns
):

    d_threshold_row = (
        threshold_df[
            threshold_df[
                "Signal"
            ]
            == "D"
        ]
        .iloc[
            0
        ]
    )


    c_threshold_row = (
        threshold_df[
            threshold_df[
                "Signal"
            ]
            == "C"
        ]
        .iloc[
            0
        ]
    )


    D_Q1 = float(
        d_threshold_row[
            "Low_Medium_Threshold"
        ]
    )


    D_Q2 = float(
        d_threshold_row[
            "Medium_High_Threshold"
        ]
    )


    C_Q1 = float(
        c_threshold_row[
            "Low_Medium_Threshold"
        ]
    )


    C_Q2 = float(
        c_threshold_row[
            "Medium_High_Threshold"
        ]
    )


else:

    state_source = pd.read_csv(
        STATE_SOURCE_PATH
    )

    D_Q1 = float(
        state_source[
            "Difficulty_D"
        ].quantile(
            1 / 3
        )
    )

    D_Q2 = float(
        state_source[
            "Difficulty_D"
        ].quantile(
            2 / 3
        )
    )

    C_Q1 = float(
        state_source[
            "Cognitive_Effort_C"
        ].quantile(
            1 / 3
        )
    )

    C_Q2 = float(
        state_source[
            "Cognitive_Effort_C"
        ].quantile(
            2 / 3
        )
    )


print(
    "\nFrozen discretisation thresholds:"
)

print(
    "D:",
    D_Q1,
    D_Q2
)

print(
    "C:",
    C_Q1,
    C_Q2
)

# 6 Prepare the gaze-guided repository

gaze_repo_str = str(
    GAZE_REPO.resolve()
)


if gaze_repo_str not in sys.path:

    sys.path.insert(
        0,
        gaze_repo_str
    )


from modeling.gaze_models import (
    CausalTransformerGazeModel
)


print(
    "\nGaze repository imported."
)



# 7 Load the pretrained gaze-guided model

GAZE_MODEL_NAME = (
    "openai-community/gpt2"
)


gaze_tokenizer = (
    AutoTokenizer.from_pretrained(
        GAZE_MODEL_NAME
    )
)


gaze_tokenizer.pad_token = (
    gaze_tokenizer.eos_token
)

gaze_model = (
    CausalTransformerGazeModel
    .from_pretrained(
        GAZE_MODEL_NAME
    )
)


checkpoint = torch.load(
    GAZE_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)


if (
    isinstance(
        checkpoint,
        dict
    )
    and
    "state_dict"
    in checkpoint
):

    state_dict = (
        checkpoint[
            "state_dict"
        ]
    )


else:

    state_dict = (
        checkpoint
    )


gaze_model.load_state_dict(
    state_dict
)


if torch.cuda.is_available():

    gaze_device = (
        torch.device(
            "cuda"
        )
    )


elif (
    hasattr(
        torch.backends,
        "mps"
    )
    and
    torch.backends.mps.is_available()
):

    gaze_device = (
        torch.device(
            "mps"
        )
    )


else:

    gaze_device = (
        torch.device(
            "cpu"
        )
    )


gaze_model.to(
    gaze_device
)

gaze_model.eval()


print(
    "Gaze model loaded successfully."
)

print(
    "Gaze device:",
    gaze_device
)

# 8 Define ReadaBERT prediction

def predict_bt_easiness(
    text
):

    encoded = (
        readability_tokenizer(
            str(
                text
            ),
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=288
        )
    )


    encoded = {
        key:
            value.to(
                readability_device
            )

        for key, value
        in encoded.items()
    }


    with torch.no_grad():

        output = (
            readability_model(
                **encoded
            )
        )


    return float(
        output
        .logits
        .squeeze()
        .cpu()
        .item()
    )


# 9 Convert predicted easiness to difficulty D

def bt_to_difficulty(
    predicted_bt
):

    normalized_easiness = (
        (
            predicted_bt
            - BT_MIN
        )
        /
        (
            BT_MAX
            - BT_MIN
        )
    )


    normalized_easiness = (
        np.clip(
            normalized_easiness,
            0.0,
            1.0
        )
    )


    difficulty = (
        1.0
        - normalized_easiness
    )


    return float(
        difficulty
    )

# 10 Define gaze prediction

def predict_gaze_raw(
    text
):

    encoded = (
        gaze_tokenizer(
            str(
                text
            ),
            return_tensors="pt",
            padding=True,
            truncation=True
        )
    )


    encoded = {
        key:
            value.to(
                gaze_device
            )

        for key, value
        in encoded.items()
    }


    with torch.no_grad():

        gaze_outputs = (
            gaze_model(
                **encoded
            )
        )


    # Remove padded-token predictions.
    gaze_outputs = (
        gaze_outputs
        * encoded[
            "attention_mask"
        ]
    )


    # Match Phase 7 passage-level aggregation.
    gaze_score = (
        gaze_outputs
        .sum(
            dim=1
        )
    )


    return float(
        gaze_score
        .cpu()
        .item()
    )

# 11 Convert raw gaze prediction to C

def gaze_to_cognitive_effort(
    gaze_raw
):

    cognitive_effort = (
        (
            gaze_raw
            - GAZE_MIN
        )
        /
        (
            GAZE_MAX
            - GAZE_MIN
        )
    )


    cognitive_effort = (
        np.clip(
            cognitive_effort,
            0.0,
            1.0
        )
    )


    return float(
        cognitive_effort
    )


# 12 Use the same discretisation logic as Phase 9/10

def d_level(
    difficulty
):

    difficulty = float(
        np.clip(
            difficulty,
            0.0,
            1.0
        )
    )


    if difficulty < D_Q1:

        return 0


    elif difficulty < D_Q2:

        return 1


    else:

        return 2


def c_level(
    cognitive_effort
):

    cognitive_effort = float(
        np.clip(
            cognitive_effort,
            0.0,
            1.0
        )
    )


    if cognitive_effort < C_Q1:

        return 0


    elif cognitive_effort < C_Q2:

        return 1


    else:

        return 2


def p_level(
    performance
):

    performance = float(
        np.clip(
            performance,
            0.0,
            1.0
        )
    )


    if performance < 1 / 3:

        return 0


    elif performance < 2 / 3:

        return 1


    else:

        return 2

# 13 Construct the complete RL state

def build_state(
    difficulty,
    cognitive_effort,
    performance
):

    return (
        d_level(
            difficulty
        ),

        c_level(
            cognitive_effort
        ),

        p_level(
            performance
        )
    )

# 14 Retrieve recommendation from frozen Phase 17 policy

def choose_haptic_feedback(
    state
):

    if state not in FINAL_POLICY:

        raise KeyError(
            f"State {state} is missing "
            "from the frozen Phase 17 policy."
        )


    action = int(
        FINAL_POLICY[
            state
        ]
    )


    return (
        action,
        HAPTIC_ACTIONS[
            action
        ]
    )

# 15 Complete end-to-end haptic system

def haptic_system(
    text,
    learner_performance
):

    if (
        text is None
        or not str(text).strip()
    ):
        raise ValueError(
            "Text cannot be empty."
        )

    performance = float(
        np.clip(
            learner_performance,
            0.0,
            1.0
        )
    )

    # Step 1:
    # Fine-tuned ReadaBERT predicts BT.Easiness.
    predicted_bt = predict_bt_easiness(
        text
    )

    # Step 2:
    # Convert predicted easiness to difficulty D.
    difficulty = bt_to_difficulty(
        predicted_bt
    )

    # Step 3:
    # Gaze-guided transformer predicts raw gaze score.
    gaze_raw = predict_gaze_raw(
        text
    )

    # Step 4:
    # Convert raw gaze score to cognitive effort C.
    cognitive_effort = gaze_to_cognitive_effort(
        gaze_raw
    )

    # Step 5:
    # Combine D, C and supplied learner performance P.
    state = build_state(
        difficulty,
        cognitive_effort,
        performance
    )

    # Step 6:
    # Retrieve recommendation from frozen Phase 17 policy.
    (
        action,
        haptic_label
    ) = choose_haptic_feedback(
        state
    )

    return {
        "Text":
            str(text),

        "Predicted_BT_Easiness":
            predicted_bt,

        "Difficulty_D":
            difficulty,

        "Gaze_Score_Raw":
            gaze_raw,

        "Cognitive_Effort_C":
            cognitive_effort,

        "Learner_Performance_P":
            performance,

        "D_Level":
            state[0],

        "D_Label":
            LEVEL_NAMES[
                state[0]
            ],

        "C_Level":
            state[1],

        "C_Label":
            LEVEL_NAMES[
                state[1]
            ],

        "P_Level":
            state[2],

        "P_Label":
            LEVEL_NAMES[
                state[2]
            ],

        "RL_State":
            state,

        "Haptic_Action":
            action,

        "Haptic_Label":
            haptic_label
    }



# 16 Define readable output function

def print_result(
    result
):

    print(
        "=" * 65
    )

    print(
        "ADAPTIVE HAPTIC FEEDBACK SYSTEM"
    )

    print(
        "=" * 65
    )

    print(
        "Predicted BT.Easiness:",
        round(
            result[
                "Predicted_BT_Easiness"
            ],
            4
        )
    )

    print(
        "Difficulty D:",
        round(
            result[
                "Difficulty_D"
            ],
            4
        ),
        "(",
        result[
            "D_Label"
        ],
        ")"
    )

    print(
        "Raw gaze score:",
        round(
            result[
                "Gaze_Score_Raw"
            ],
            4
        )
    )

    print(
        "Cognitive Effort C:",
        round(
            result[
                "Cognitive_Effort_C"
            ],
            4
        ),
        "(",
        result[
            "C_Label"
        ],
        ")"
    )

    print(
        "Learner Performance P:",
        round(
            result[
                "Learner_Performance_P"
            ],
            4
        ),
        "(",
        result[
            "P_Label"
        ],
        ")"
    )

    print(
        "RL State:",
        result[
            "RL_State"
        ]
    )

    print(
        "Recommended Haptic Feedback:",
        result[
            "Haptic_Label"
        ]
    )

    print(
        "=" * 65
    )

# 17 Run one complete end-to-end example

example_text = (
    "Photosynthesis is the process through which plants "
    "convert light energy into chemical energy."
)


example_result = haptic_system(
    text=example_text,
    learner_performance=0.50
)


print(
    "\nEND-TO-END EXAMPLE"
)

print_result(
    example_result
)


# Save the complete example.
example_output_df = pd.DataFrame(
    [
        example_result
    ]
)


example_output_df[
    "RL_State"
] = (
    example_output_df[
        "RL_State"
    ]
    .astype(
        str
    )
)


example_output_df.to_csv(
    PROTOTYPE_OUTPUT_PATH,
    index=False
)


print(
    "\nSaved end-to-end example:",
    PROTOTYPE_OUTPUT_PATH
)



# 18 RQ5 personalisation test

performance_profiles = {
    "Low performance": 0.20,
    "Medium performance": 0.50,
    "High performance": 0.80
}


personalisation_results = []


for profile_name, performance in (
    performance_profiles.items()
):

    result = haptic_system(
        text=example_text,
        learner_performance=performance
    )

    personalisation_results.append(
        {
            "Performance_Profile":
                profile_name,

            "Learner_Performance_P":
                performance,

            "Predicted_BT_Easiness":
                result[
                    "Predicted_BT_Easiness"
                ],

            "Difficulty_D":
                result[
                    "Difficulty_D"
                ],

            "D_Level":
                result[
                    "D_Level"
                ],

            "D_Label":
                result[
                    "D_Label"
                ],

            "Gaze_Score_Raw":
                result[
                    "Gaze_Score_Raw"
                ],

            "Cognitive_Effort_C":
                result[
                    "Cognitive_Effort_C"
                ],

            "C_Level":
                result[
                    "C_Level"
                ],

            "C_Label":
                result[
                    "C_Label"
                ],

            "P_Level":
                result[
                    "P_Level"
                ],

            "P_Label":
                result[
                    "P_Label"
                ],

            "RL_State":
                str(
                    result[
                        "RL_State"
                    ]
                ),

            "Haptic_Action":
                result[
                    "Haptic_Action"
                ],

            "Haptic_Label":
                result[
                    "Haptic_Label"
                ]
        }
    )


personalisation_df = pd.DataFrame(
    personalisation_results
)


print(
    "\nRQ5 PERSONALISATION TEST"
)

print(
    "------------------------"
)

print(
    personalisation_df[
        [
            "Performance_Profile",
            "Learner_Performance_P",
            "Difficulty_D",
            "D_Label",
            "Cognitive_Effort_C",
            "C_Label",
            "P_Label",
            "RL_State",
            "Haptic_Label"
        ]
    ]
    .to_string(
        index=False
    )
)

# 19 Verify that only P changed

D_VALUES_EQUAL = (
    np.allclose(
        personalisation_df[
            "Difficulty_D"
        ],
        personalisation_df[
            "Difficulty_D"
        ].iloc[
            0
        ]
    )
)


C_VALUES_EQUAL = (
    np.allclose(
        personalisation_df[
            "Cognitive_Effort_C"
        ],
        personalisation_df[
            "Cognitive_Effort_C"
        ].iloc[
            0
        ]
    )
)


BT_VALUES_EQUAL = (
    np.allclose(
        personalisation_df[
            "Predicted_BT_Easiness"
        ],
        personalisation_df[
            "Predicted_BT_Easiness"
        ].iloc[
            0
        ]
    )
)


GAZE_VALUES_EQUAL = (
    np.allclose(
        personalisation_df[
            "Gaze_Score_Raw"
        ],
        personalisation_df[
            "Gaze_Score_Raw"
        ].iloc[
            0
        ]
    )
)


print(
    "\nPERSONALISATION CONTROL CHECK"
)

print(
    "Same text:",
    True
)

print(
    "BT prediction unchanged:",
    BT_VALUES_EQUAL
)

print(
    "Difficulty D unchanged:",
    D_VALUES_EQUAL
)

print(
    "Raw gaze prediction unchanged:",
    GAZE_VALUES_EQUAL
)

print(
    "Cognitive effort C unchanged:",
    C_VALUES_EQUAL
)


if not (
    D_VALUES_EQUAL
    and C_VALUES_EQUAL
    and BT_VALUES_EQUAL
    and GAZE_VALUES_EQUAL
):

    raise ValueError(
        "D or C changed even though "
        "the same text was used."
    )


print(
    "Control check passed: "
    "only learner performance P changed."
)

# 20 Check whether the frozen policy personalised

unique_actions = (
    personalisation_df[
        "Haptic_Action"
    ]
    .nunique()
)


unique_labels = (
    personalisation_df[
        "Haptic_Label"
    ]
    .nunique()
)


print(
    "\nNumber of different haptic actions:",
    unique_actions
)


print(
    "Number of different haptic labels:",
    unique_labels
)


if unique_actions > 1:

    PERSONALISATION_DEMONSTRATED = (
        True
    )

    print(
        "\nPERSONALISATION DEMONSTRATED:"
    )

    print(
        "The same educational text produced "
        "different haptic recommendations "
        "when only learner performance P changed."
    )


else:

    PERSONALISATION_DEMONSTRATED = (
        False
    )

    print(
        "\nNO ACTION CHANGE FOR THIS PARTICULAR TEXT:"
    )

    print(
        "The pipeline ran correctly, but the three "
        "performance levels mapped to the same haptic "
        "action for this D-C combination."
    )

# 21 Save the RQ5 personalisation demonstration

personalisation_df[
    "Personalisation_Demonstrated"
] = (
    PERSONALISATION_DEMONSTRATED
)


personalisation_df.to_csv(
    DEMO_OUTPUT_PATH,
    index=False
)


print(
    "\nSaved personalisation test:",
    DEMO_OUTPUT_PATH
)

# 22 Inspect the relevant frozen-policy states directly

example_d_level = (
    int(
        personalisation_df[
            "D_Level"
        ].iloc[
            0
        ]
    )
)


example_c_level = (
    int(
        personalisation_df[
            "C_Level"
        ].iloc[
            0
        ]
    )
)


print(
    "\nFROZEN POLICY FOR THIS TEXT'S D-C COMBINATION"
)

print(
    "------------------------------------------------"
)


for performance_state in [
    0,
    1,
    2
]:

    state = (
        example_d_level,
        example_c_level,
        performance_state
    )

    action = (
        FINAL_POLICY[
            state
        ]
    )

    print(
        "D =",
        LEVEL_NAMES[
            example_d_level
        ],
        "| C =",
        LEVEL_NAMES[
            example_c_level
        ],
        "| P =",
        LEVEL_NAMES[
            performance_state
        ],
        "->",
        HAPTIC_ACTIONS[
            action
        ]
    )


# 23 Final Phase 19 integrity checks

assert len(
    FINAL_POLICY
) == 27


assert (
    personalisation_df[
        "Learner_Performance_P"
    ]
    .nunique()
    == 3
)


assert (
    personalisation_df[
        "Difficulty_D"
    ]
    .nunique()
    == 1
)


assert (
    personalisation_df[
        "Cognitive_Effort_C"
    ]
    .nunique()
    == 1
)


print(
    "\nPhase 19 integrity checks passed."
)

# 24 Final Phase 19 summary

print(
    "\nPHASE 19 FINAL SUMMARY"
)

print(
    "----------------------"
)


print(
    "Input text:"
)

print(
    example_text
)


print(
    "\nPredicted BT.Easiness:",
    round(
        example_result[
            "Predicted_BT_Easiness"
        ],
        4
    )
)


print(
    "Difficulty D:",
    round(
        example_result[
            "Difficulty_D"
        ],
        4
    ),
    "-",
    example_result[
        "D_Label"
    ]
)


print(
    "Cognitive Effort C:",
    round(
        example_result[
            "Cognitive_Effort_C"
        ],
        4
    ),
    "-",
    example_result[
        "C_Label"
    ]
)


print(
    "\nPerformance-dependent recommendations:"
)


for _, row in (
    personalisation_df.iterrows()
):

    print(
        row[
            "Performance_Profile"
        ],
        "(",
        row[
            "P_Label"
        ],
        ")",
        "->",
        row[
            "Haptic_Label"
        ]
    )


print(
    "\nPersonalisation demonstrated:",
    PERSONALISATION_DEMONSTRATED
)

# 25 Confirm Phase 19 completion

print(
    "\nPHASE 19 COMPLETE"
)

print(
    "End-to-end adaptive haptic prototype "
    "executed successfully."
)

print(
    "Frozen policy used:",
    POLICY_PATH
)

print(
    "Saved end-to-end example:",
    PROTOTYPE_OUTPUT_PATH
)

print(
    "Saved RQ5 personalisation test:",
    DEMO_OUTPUT_PATH
)


Python: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/readabert_env/bin/python
Working directory: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++

PHASE 19 ASSET CHECK
-----------------------------------------------------------------
Final Phase 17 policy       : FOUND
    /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/phase17_final_haptic_policy_rerun.csv
Fine-tuned ReadaBERT        : FOUND
    /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/readabert_clear_finetuned_best_rerun
CLEAR training data         : FOUND
    /Users/nataliakolbina/Documents/workspace/haptic/CLEAR_original_grouped_split_2026-09-25/CLEAR_original_train.csv
Gaze repository             : FOUND
    /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/gaze-guided-text-generation
Gaze checkpoint             : FOUND
    /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/gaze-